# Race day

**Assemble, qualify and submit.**

The real race is one track, CARLA's Monza, three laps, raced by one file.
You know the track: racekit's stand-in for it is `monza-like`, and you have raced it since Unit 7.
You do not know the car: CARLA's car is not racekit's carla-like car, and no model of a car is exact (Unit 11).
So race day is every tool of the book, in one order: assemble the parts the units built, re-tune them on the known track against cars you have not met, keep a margin, choose the steering, check the file, and submit it.

`agents/unit-13.py` is that file, the book's race-day agent for `monza-like`.
On race day it has to pass four things (design 013's gate for this unit):

- `check_submission` on its own track, with `step()`'s 99th percentile at most 5 ms;
- every race clean: 5 of 5 on the nominal car, and 24 of 24 on each set of cars it has never been tuned on;
- faster than Part 1's agent, `agents/unit-07.py`, by Unit 1's rule;
- the packaging checklist, so that the file you submit is the file you tested.

Here are the two agents, live: `agents/unit-07.py` (Unit 7's speed profile on the centreline, no line and no measured grip) and `agents/unit-13.py`, on `monza-like`, 3 laps, race seed 0, the carla-like car with its noise (about 15 s):

In [ ]:
import math
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
import racekit
from racekit import write_variant
from racekit.cars import carla_like, perturbation_factors
from racekit.check_submission import check_submission
from racekit.measure import noise_aware_better
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
UNIT_07, UNIT_11, UNIT_13 = AGENTS / "unit-07.py", AGENTS / "unit-11.py", AGENTS / "unit-13.py"
workdir = Path(mkdtemp(prefix="roar-unit-13-"))
TRACK = "monza-like"  # the real race's track is known: racekit's stand-in for CARLA's Monza
TICK = 0.05 + 1e-9  # s: race times are pinned to one control tick
DT = 0.05  # s: one control tick
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
CAP = 900.0  # s: a monza-like race is cut here
NO_TIMING = {"MPC_BUDGET_MS": None}  # every race here: no wall clock inside the race (Unit 12, trap 4)
MONZA_MU_G = (5.38, 8.96, 9.12, 9.14, 9.70, 7.97)  # m/s^2: Unit 8's six grip probes on monza-like

# Unit 9's line solver: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x

NAME_CHARACTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_"


def constant_name(line):
    # The UPPER_CASE name of a module-constant line ("NAME = value  # comment"), or None (Unit 12's rule).
    cuts = [i for i in range(len(line) - 2) if line[i:i + 3] == " = "]
    if cuts and line[0] in NAME_CHARACTERS[:26] and all(c in NAME_CHARACTERS for c in line[:cuts[0]]):
        return line[:cuts[0]]
    return None


def module_constants(path):
    # name -> the value's text (after " = ", without a "  # ..." comment), for every module-constant line.
    # A value written over several lines (LINE_CONTROL) is only its first line here: "(".
    constants = {}
    with open(path) as handle:
        for line in handle:
            name = constant_name(line)
            if name is not None:
                text = line[len(name) + 3:-1]
                comment = [i for i in range(len(text) - 2) if text[i:i + 3] == "  #"]
                constants[name] = text[:comment[0]] if comment else text
    return constants


def file_parts(path):
    # (the comment lines above the docstring, the rest of the file without its constant lines and without
    # the line's values): the header may change, the rest must not.
    with open(path) as handle:
        lines = [line for line in handle]
    start = [i for i, line in enumerate(lines) if line[:3] == '"""'][0]
    body, in_line = [], False
    for line in lines[start:]:
        if line[:15] == "LINE_CONTROL = ":
            in_line = True
        elif in_line:
            in_line = line != ")\n"
        elif constant_name(line) is None:
            body.append(line)
    return lines[:start], body


track = load_track(TRACK)
with shim_installed():
    Unit11, name11 = load_submission(UNIT_11)
    Unit13, name13 = load_submission(UNIT_13)
eleven, thirteen = sys.modules[name11], sys.modules[name13]  # the two files' constants and functions
agent13 = Unit13(track.waypoints, None)
LINE = np.array(agent13.offsets)  # m: unit-13.py's racing line, as offsets from the centreline
CENTRELINE = np.zeros(len(track))  # unit-07.py drives the centreline
written = []  # every variant file written so far


def variant(source, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, each written as its repr.
    path = workdir / f"v{len(written):03d}.py"
    written.append(path)
    return write_variant(source, path, {key: repr(value) for key, value in constants.items()})


def race(source=UNIT_13, seed=0, k=None, laps=3, **constants):
    # One race on monza-like of `source` with `constants` changed and the timing off, on the carla-like car with
    # its noise (car k of Unit 11's perturbed family if k is given). unit-07.py has no clock: it races as shipped.
    path = source if source == UNIT_07 else variant(source, **constants, **NO_TIMING)
    return run_race(path, TRACK, car=carla_like(noise=True, perturb=k), seed=seed, laps=laps, max_seconds=CAP)


def error_trace(result, offsets=LINE):
    # Unit 9: the time and the cross-track error from the race's own line over the driving rows (held == 0).
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return telemetry["t"][driving], telemetry["offset"][driving] - target


def line_error(result, offsets=LINE):
    return float(np.max(np.abs(error_trace(result, offsets)[1])))


def clean(result, offsets=LINE):
    return result.finished and result.collisions == 0 and line_error(result, offsets) <= CLEAN_ERROR


def show(label, result, offsets=LINE):
    print(f"{label:28s} {result.total_time:7.2f} s, {result.collisions} collisions, "
          f"{line_error(result, offsets):.2f} m from its line, clean {clean(result, offsets)}")


seven = race(UNIT_07)
show("unit-07.py, seed 0", seven, CENTRELINE)
hook = race(UNIT_13)
show("unit-13.py, seed 0", hook)
print(f"unit-13.py is {seven.total_time - hook.total_time:.2f} s faster over 3 laps")
assert clean(seven, CENTRELINE) and abs(seven.total_time - 515.10) <= TICK  # p1
assert clean(hook) and abs(hook.total_time - 468.55) <= TICK  # U13.1, seed 0

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(seven.telemetry["t"], seven.telemetry["v"], linewidth=0.6, label="unit-07.py (Part 1)")
ax.plot(hook.telemetry["t"], hook.telemetry["v"], linewidth=0.6, label="unit-13.py (race day)")
ax.set_xlabel("t (s)")
ax.set_ylabel("speed (m/s)")
ax.set_title("monza-like, 3 laps, race seed 0")
ax.legend()
plt.show()

46.55 s over three laps, both clean.
The speed trace shows where it comes from: `unit-13.py` carries more speed through the corners.
It plans on a racing line (Unit 9) with 64% of the grip it measured (Unit 8), 5.1 to 6.2 m/s² of cornering in the sections with corners, where `unit-07.py` plans on the centreline with 4.6 m/s² everywhere, and it accelerates harder out of them (`A_ACC` 5.0 against 4.0).
One race at one seed is not the gate: the milestone races both agents at seeds 0 to 4 and applies Unit 1's rule.

This unit **introduces nothing**.
Every step below is a tool you already have, used on the race-day question:

- §1 **assemble**: which part of `unit-13.py` each unit built, and one measurement you can trust only because of the line's shape;
- §2 **the known track and the unknown car**: Unit 12's crash-aware re-tune, on `monza-like` itself, and a floating-point trap it fell into;
- §3 **the margin**, on every knob at its edge, checked on cars the search never saw;
- §4 **MPC or pursuit**, Unit 10's question again;
- §5 **qualify**: `check_submission`, and a timing policy you switch off on purpose;
- §6 **package and submit**: the checklist, and what the season's rules say.

Every race in this notebook runs with the timing off.
Every number that is not raced here is from the probe record of plan `roar-013` and says so.

## 1. Assemble

`agents/unit-13.py` is `agents/unit-11.py` with six module constants changed, and nothing else.
Every part of it was built by a unit:

| part | unit | in `unit-13.py` |
|---|---|---|
| the waypoints, the progress, the clean rule | Units 1, 2 | as in `unit-11.py` |
| the speed loop (a PI controller) | Unit 4 | as in `unit-11.py` |
| the speed profile (the backward and forward passes) | Unit 7 | `A_DEC`, `A_ACC` **re-tuned** (§2, §3) |
| the measured grip, μ·g per section | Unit 8 | `MU_G`: `monza-like`'s six probes |
| the racing line | Unit 9 | `LINE_CONTROL`: `monza-like`'s line |
| MPC steering and its tick budget | Unit 10 | `MPC_MIN_HORIZON` = `MPC_HORIZON` (§5) |
| the respawn fallback, the margin rule | Unit 11 | the fraction `MU_FRACTION` **re-tuned** (§3) |
| the crash-aware objective, the packaging checklist | Unit 12 | §2 and §6 |

The cell reads both files as text with Unit 12's helper (each constant's value without its comment), compares them, checks that everything else in the file is the same line for line, and solves `monza-like`'s line again with Unit 9's solver (about 2 s):

In [ ]:
eleven_text, thirteen_text = module_constants(UNIT_11), module_constants(UNIT_13)
changed = [name for name in eleven_text if eleven_text[name] != thirteen_text[name]]
for name in changed:
    print(f"{name:16s} unit-11.py {eleven_text[name]:40s} unit-13.py {thirteen_text[name]}")
print(f"{'LINE_CONTROL':16s} unit-11.py {len(eleven.LINE_CONTROL)} values (test-circuit)"
      f"{'':19s} unit-13.py {len(thirteen.LINE_CONTROL)} values ({TRACK})")
header13, body13 = file_parts(UNIT_13)
header11, body11 = file_parts(UNIT_11)
print(f"unit-13.py: {len(header13)} header comment lines; the other {len(body13)} lines (code, docstring, imports) "
      f"the same as unit-11.py's: {body13 == body11}")
assert set(eleven_text) == set(thirteen_text) and changed == ["A_DEC", "A_ACC", "MU_G", "MU_FRACTION", "MPC_MIN_HORIZON"]
assert len(eleven.LINE_CONTROL) == 133 and len(thirteen.LINE_CONTROL) == 285  # U13.0: six constants in all
assert body13 == body11 and header11 == [] and all(line[0] == "#" for line in header13)
assert thirteen.MU_G == MONZA_MU_G and thirteen.MPC_MIN_HORIZON == thirteen.MPC_HORIZON == 40

# Unit 9: monza-like's line, solved here, against the 285 values the file ships (each written with repr).
control = solve_line(track)
difference = float(np.max(np.abs(control - np.array(thirteen.LINE_CONTROL))))
print(f"{TRACK}: {len(track)} waypoints, {len(control)} control values; the largest |solved - shipped| {difference:.1e} m")
assert len(control) == 285 and np.allclose(control, thirteen.LINE_CONTROL, rtol=0.0, atol=1e-6)  # U13.0

Five constants differ in their text, and the sixth is the line: 133 values solved for `test-circuit` in `unit-11.py`, 285 solved for `monza-like` here.
Everything else, every function, the docstring and the imports, is `unit-11.py` line for line; only the comment lines at the top (the header, which says what changed and why) are new.
And the line solved in this notebook is the line in the file.

**Section 0's grip is a lower bound.**
Unit 8 measures a section's grip by pushing the car until it slides there.
On `monza-like` the probe of section 0 never slid, so its 5.38 m/s² is only "at least 5.38" (the other five sections measured 7.97 to 9.70).
Does it matter?
Unit 7's corner cap answers without a race: through a corner of curvature κ the car can go at most v_max = √(μg / κ), and the agent plans with `MU_FRACTION` of that grip and never above `TARGET_SPEED`.
The agent you built in the setup cell knows the curvature and the section of every point of its line:

In [ ]:
curvature = np.abs(np.array(agent13.curvature))  # 1/m, on the racing line
section = np.array(agent13.section)
fraction, top = thirteen.MU_FRACTION, thirteen.TARGET_SPEED
print("section  largest |kappa| (1/m)  radius (m)   v_max at mu*g (m/s)   the planner's cap (m/s)")
for s in range(6):
    kappa = float(np.max(curvature[section == s]))
    v_max = math.sqrt(thirteen.MU_G[s] / kappa)
    cap = min(top, math.sqrt(fraction * thirteen.MU_G[s] / kappa))
    print(f"{s:7d}  {kappa:21.2e}  {1 / kappa:10.0f}   {v_max:19.1f}   {cap:22.1f}")
kappa_0 = float(np.max(curvature[section == 0]))
print(f"section 0's cap drops below {top:.0f} m/s only if its grip is below {top ** 2 * kappa_0 / fraction:.3f} m/s^2")
assert kappa_0 < 3e-5 and math.sqrt(fraction * thirteen.MU_G[0] / kappa_0) > 300.0

# The probe record of plan roar-013 (p5): unit-13.py's setting with section 0's grip changed, nominal car,
# race seeds 0-4, 3 laps. (median race time in s, max line error over the five races in m)
P5 = {5.38: (468.55, 1.009), 7.97: (468.55, 1.009), 9.12: (468.55, 1.009), 12.0: (468.55, 1.009)}
for mu_g, (median, error) in P5.items():
    print(f"MU_G[0] = {mu_g:5.2f}: median {median:.2f} s, max line error {error:.3f} m")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.semilogy(range(len(curvature)), np.clip(curvature, 1e-6, None), linewidth=0.6)
for s in range(1, 6):
    ax.axvline(int(np.argmax(section == s)), color="grey", linestyle="--", linewidth=1.0)
ax.set_xlabel("waypoint (dashed: the start of grip sections 1 to 5)")
ax.set_ylabel("|curvature| on the line (1/m)")
ax.set_title("monza-like: the curvature of the racing line, section by section")
plt.show()

Section 0's sharpest point on the line has a radius of about 45 km: on this line it is a straight.
At any grip above 0.07 m/s² its cap is the top speed, 45 m/s, so a lower bound of 5.38 costs nothing, and the record agrees: the same races, to the tick and to the millimetre, whether section 0's grip is written as 5.38, 7.97, 9.12 or 12.0.
The other five sections all cap the speed well below 45 m/s: there the measured grip is the race.

That reasoning is about **this line on this track**.
A lower bound in a section with a real corner would be a real cost (the car would brake for a grip it has), and on a different line the same section might not be a straight.

## 2. The known track and the unknown car

Unit 12 kept tracks apart: training tracks for the search, a validation track to catch overfitting, a test track raced once.
Race day turns that around.
The track is **known**: the real race is Monza, and tuning on `monza-like` is not cheating, it is the job.
What you cannot tune on is the **car**.
So the held-out things here are cars: Unit 11's perturbed family, `carla_like(perturb=k)`, each car a few per cent off in grip, steering, throttle, brakes, mass and drag, raced at race seed k mod 5.

| cars | role in this unit |
|---|---|
| k = 0 to 23 | **training**: the objective races the three with the lowest grip, 16, 18 and 8 |
| k = 24 to 71 | **validation**: they chose between settings already found (§3), so they are spent |
| k = 72 to 95 | **test**: chosen before anything raced on them, raced **once**, with the shipped file |

**The objective** is Unit 12's crash-aware one, on the known track.
Each evaluation is five races of 3 laps on `monza-like`: the nominal car at race seeds 0 and 1, and cars 16, 18 and 8.
Its two **per-replicate sums** are the nominal race at seed r plus the three cars' times (r = 0, 1).
Any race that is not clean rejects the setting; Unit 1's rule on the two sums accepts a step.
The search is a coordinate search from `unit-11.py`'s (0.54, 5.0, 4.0) over `MU_FRACTION` (grid 0.02), `A_DEC` (0.5) and `A_ACC` (0.5), exactly as Unit 12's.
The probe record of plan `roar-013` (p2) ran it: 100 races, 20 settings, about 15 minutes of computer time.

In [ ]:
grips = {k: perturbation_factors(k)["grip"] for k in range(24)}  # relative to the carla-like car (1.0)
lowest = sorted(grips, key=lambda k: grips[k])[:3]
print("the three lowest-grip training cars:", ", ".join(f"car {k} (grip x {grips[k]:.3f})" for k in lowest))
assert lowest == [16, 18, 8]

# The probe record of plan roar-013 (p2), in the order the search evaluated them:
# (MU_FRACTION, A_DEC, A_ACC) -> ((the two per-replicate sums in s, each exactly as the search computed it, written
# with repr), {car: (collisions, max line error in m)} for every race that was not clean).
P2 = {
    (0.54, 5.0, 4.0): ((1929.9, 1929.8500000000001), {}),  # agents/unit-11.py
    (0.56, 5.0, 4.0): ((1916.3000000000002, 1916.3000000000002), {}),
    (0.52, 5.0, 4.0): ((1943.9, 1943.75), {}),
    (0.58, 5.0, 4.0): ((1905.7, 1905.65), {}),
    (0.6, 5.0, 4.0): ((1895.5500000000002, 1895.5000000000002), {}),
    (0.62, 5.0, 4.0): ((1886.0, 1885.95), {}),
    (0.64, 5.0, 4.0): ((1876.8500000000001, 1876.8000000000002), {}),
    (0.66, 5.0, 4.0): ((1868.4499999999998, 1868.3999999999999), {}),
    (0.68, 5.0, 4.0): ((1959.6000000000001, 1959.5500000000002), {16: (0, 2.012), 8: (1, 4.510)}),
    (0.66, 5.5, 4.0): ((2130.6000000000004, 2130.6000000000004), {16: (0, 2.794), 18: (0, 2.805), 8: (2, 3.150)}),
    (0.66, 4.5, 4.0): ((1875.7500000000002, 1875.7500000000002), {}),
    (0.66, 5.0, 4.5): ((1862.5500000000002, 1862.6000000000001), {}),
    (0.66, 5.0, 3.5): ((1880.75, 1880.7), {}),
    (0.66, 5.0, 5.0): ((1861.95, 1861.9), {}),
    (0.66, 5.0, 5.5): ((1861.9, 1861.8500000000001), {}),
    (0.66, 5.0, 6.0): ((1862.0500000000002, 1861.95), {}),
    (0.68, 5.0, 5.5): ((2491.6, 2491.45), {16: (2, 6.846), 8: (2, 7.717)}),
    (0.64, 5.0, 5.5): ((1870.3000000000002, 1870.3000000000002), {}),
    (0.66, 5.5, 5.5): ((1856.15, 1856.0500000000002), {16: (0, 2.815), 18: (0, 2.804), 8: (0, 2.780)}),
    (0.66, 4.5, 5.5): ((1869.5000000000002, 1869.5000000000002), {}),
}
TAKEN = [(0.54, 5.0, 4.0), (0.56, 5.0, 4.0), (0.58, 5.0, 4.0), (0.6, 5.0, 4.0), (0.62, 5.0, 4.0), (0.64, 5.0, 4.0),
         (0.66, 5.0, 4.0), (0.66, 5.0, 4.5), (0.66, 5.0, 5.0), (0.66, 5.0, 5.5)]  # the search's steps, in order
for setting, (sums, bad) in P2.items():
    why = "; ".join(f"car {k}: {n} collisions, {e:.2f} m" for k, (n, e) in bad.items())
    print(f"{setting}: {sums[0]:8.2f} / {sums[1]:8.2f} s  {'taken' if setting in TAKEN else '     '}  "
          f"{'clean' if not bad else 'NOT clean: ' + why}")

# The search's last step: A_ACC 5.0 -> 5.5.
before, after = P2[(0.66, 5.0, 5.0)][0], P2[(0.66, 5.0, 5.5)][0]
gain = float(np.median(before) - np.median(after))
noise = max(max(before) - min(before), max(after) - min(after))
print(f"A_ACC 5.0 -> 5.5: the gain {gain!r} s against the larger spread {noise!r} s; "
      f"Unit 1's rule says 5.5 beats 5.0: {noise_aware_better(after, before)}")
before_ticks, after_ticks = [round(t / DT) for t in before], [round(t / DT) for t in after]
print(f"in whole ticks: {before_ticks} -> {after_ticks}; Unit 1's rule: {noise_aware_better(after_ticks, before_ticks)}")
assert noise_aware_better(after, before) and not noise_aware_better(after_ticks, before_ticks)

fig, ax = plt.subplots(figsize=(9, 3.5))
clean_order = [(i, float(np.median(sums))) for i, (sums, bad) in enumerate(P2.values()) if not bad]
ax.plot([i for i, _ in clean_order], [value for _, value in clean_order], "o", label="a clean evaluation")
position = {setting: i for i, setting in enumerate(P2)}
ax.step([position[s] for s in TAKEN], [float(np.median(P2[s][0])) for s in TAKEN], where="post",
        label="the incumbent")
ax.set_xlabel("evaluation (5 races each; the 5 not clean are not drawn)")
ax.set_ylabel("median per-replicate sum (s)")
ax.set_title("p2: the crash-aware re-tune on monza-like")
ax.legend()
plt.show()

`MU_FRACTION` climbed clean from 0.54 to 0.66, 61 s faster over the five races, and 0.68 is **not clean**: car 8 collides and car 16 runs 2.01 m from its line.
`A_DEC` 5.5 is not clean on all three low-grip cars (4.5 is clean but slower), so it never moved.
`A_ACC` went 4.0 → 4.5 → 5.0 → 5.5, and then 6.0 was no better.

**The last step is a floating-point tie.**
Between `A_ACC` 5.0 and 5.5 the sums are [1861.95, 1861.90] and [1861.90, 1861.85]: each 5.5 race sum is one tick faster, and each pair spreads by one tick.
A gain of one tick against a spread of one tick is not "more than the noise".
But the sums are floats, and `1861.9 - 1861.8500000000001` is a hair less than 0.05 while the gap between the medians is a hair more, so the float comparison says "beats".
That is Unit 12's trap 3 (a race time is a whole number of ticks; never compare computed sums with `>` or `==` as if they were exact), caught in a real search: decided in whole ticks the step is a tie, and the search stops at 5.0.

So the incumbent is **(0.66, 5.0, 5.0)**, and it sits on an **edge** in two knobs:

- `MU_FRACTION`: 0.68 is not clean;
- `A_DEC`: 5.5 is not clean.

Both edges were found at other values of `A_ACC` (4.0 and 5.5); the record (p10) raced both again at `A_ACC` 5.0, and both are still not clean on all three low-grip cars there.
`A_ACC` has **no edge** the search found: 6.0 is clean, and 6.5 or 7.0 were never raced.
Exercise 3 replays the whole search from the table.

## 3. The margin, on every knob at its edge

Unit 11's **margin rule**: the edge is the highest value that is clean; ship one grid step inside it.
Unit 11 stated it for its one knob, the fraction.
Here two knobs are at an edge, and there are two ways to read the rule:

- **M**, the margin on Unit 11's knob only: (0.64, 5.0, 5.0);
- **M2**, the margin on **every knob at its edge**: (0.64, 4.5, 5.0).

`A_ACC` gets no margin in either: a knob that was never pushed to a setting that is not clean has no edge to step back from.

The search's objective raced the nominal car and three training cars.
To see what a setting does on cars it never saw, race it on the **held-out** cars, k = 24 to 47.
Here is one of them, car 41 (race seed 1), live, with M and with M2 (`unit-13.py` as shipped) (about 25 s: M's race runs 686 s of race time):

In [ ]:
margin_one = race(UNIT_13, seed=41 % 5, k=41, A_DEC=5.0)  # M: (0.64, 5.0, 5.0)
margin_all = race(UNIT_13, seed=41 % 5, k=41)  # M2: unit-13.py as shipped, (0.64, 4.5, 5.0)
show("M, car 41", margin_one)
show("M2, car 41", margin_all)
print("M's collision at", ", ".join(f"{t:.2f} s" for t in margin_one.respawn_times))
assert not clean(margin_one) and margin_one.collisions == 1  # U13.3
assert clean(margin_all) and abs(margin_all.total_time - 467.40) <= TICK  # p3

# The probe record of plan roar-013 (p3, p3b, p3c): every setting on the 24 held-out cars k = 24-47,
# 3 laps, race seed k mod 5. (clean of 24, median in s, max line error in m, the races that were not clean)
HELD_OUT = {
    "E, the edge (0.66, 5.0, 5.0)": (22, 465.05, 9.53, "car 24 (2 collisions, 737 s), car 41 (2 collisions, 889 s)"),
    "E', the search's literal output (0.66, 5.0, 5.5)": (22, 465.00, 2.64, "cars 41 and 46 (1 collision each)"),
    "M, margin on MU_FRACTION only (0.64, 5.0, 5.0)": (23, 467.02, 5.42, "car 41 (1 collision, 686 s)"),
    "M2, margin on both edged knobs (0.64, 4.5, 5.0)": (24, 468.83, 1.34, ""),
    "P, margin on A_DEC only (0.66, 4.5, 5.0)": (24, 466.73, 1.44, ""),
    "unit-11.py as shipped (0.54, 5.0, 4.0)": (24, 482.23, 1.31, ""),
    "unit-12.py as shipped (0.54, 5.5, 4.0)": (24, 480.48, 1.87, ""),
    "unit-07.py as shipped": (12, 708.17, 6.73, "12 cars, 144 collisions, each cut at 900 s"),
}
for name, (n_clean, median, error, bad) in HELD_OUT.items():
    print(f"{name:50s} {n_clean:2d}/24 clean, median {median:6.2f} s, max {error:4.2f} m  {bad}")

# The probe record (p8, p11): M2 on fresh cars, after the choice was made.
FRESH = {"cars 48-71 (p8; raced again in p9: validation)": (24, 469.65, 1.37),
         "cars 72-95 (p11; the test cars, raced once)": (24, 468.65, 1.37)}
for name, (n_clean, median, error) in FRESH.items():
    print(f"M2 on {name:47s} {n_clean}/24 clean, median {median:.2f} s, max {error:.2f} m")

fig, ax = plt.subplots(figsize=(10, 3.5))
for label, result in (("M (0.64, 5.0, 5.0)", margin_one), ("M2 (0.64, 4.5, 5.0)", margin_all)):
    t, error = error_trace(result)
    ax.plot(t, error, linewidth=0.6, label=label)
for limit in (-CLEAN_ERROR, CLEAN_ERROR):
    ax.axhline(limit, color="grey", linestyle="--", linewidth=1.0)
ax.set_xlabel("t (s)")
ax.set_ylabel("error from the line (m)")
ax.set_title("held-out car 41, race seed 1: the 2 m rule dashed")
ax.legend()
plt.show()

On car 41, M leaves its line and hits the wall about 208 s in, and the respawn costs it a whole race: 686 s, where M2 finishes clean in 467.40 s.
The table says the same thing over all 24 held-out cars:

- the edge E (and the search's literal output E′) crash on 2 cars;
- the margin on `MU_FRACTION` alone, M, still crashes on one, car 41: Unit 11's rule, read too narrowly, is **not** enough;
- the margin on **every knob at its edge**, M2, is clean on all 24, with its line error at most 1.34 m;
- Part 1's `unit-07.py`, which never measured anything, crashes in a loop on half of the cars.

**The unit ships M2: `agents/unit-13.py`.**
It was the rule's answer before any held-out race: the record raced E, E′, M and M2 in one job declared beforehand.
It is also clean on all 24 **training** cars (median 468.35 s).

**Which cars are still a test?**
The held-out cars 24 to 47 chose M2 over M, so, as Unit 12 found for tracks, they have taken part in a choice: they are now a **validation** set.
Cars 48 to 71 were set aside as a test, raced M2 once (24/24 clean), and then were raced again under other timing settings (§5): validation too.
So the record set aside fresh cars, 72 to 95, **after** the shipped file was fixed, and raced them **once** with it: 24/24 clean, median 468.65 s.
Those are the unit's test cars; the routed premise is the only thing that races them again, to prove the result reproduces.
And all three sets are racekit's stand-in for "a car you have not met".
The real test is CARLA's car, in this unit's lab.

**Would expected race time pick differently?**
Unit 11 also ranked settings by their **expected** (mean) race time over a family, a crash counted at its cost.
Over the held-out cars (a race cut at 900 s counted at 900 s) the record gives E 494.11 s, M 476.24 s, M2 468.98 s and `unit-07.py` 707.62 s.
Here expected time **also** picks M2: E saves about 3.9 s in a clean race, and two crashes in 24 races cost far more than that.
That is not what happened in Checkpoint B, where expected time picked a fraction past the family's edge; Exercise 6 finds the crash rate at which the two rules would part.

P, the margin only on `A_DEC`, is also 24/24 clean and about 2 s faster than M2 on the held-out cars.
Is that a finding, or a fit to the cars you looked at?
That is Challenge 9.

## 4. MPC or pursuit

Unit 10 asked whether MPC is worth its cost, and the answer was margin, not speed.
Ask it again, at the race-day setting: `unit-13.py`, and the same file with `STEERING = "pursuit"`.
The probe record of plan `roar-013` (p4, p3c), and pursuit on held-out car 34 (race seed 4), live (about 5 s):

In [ ]:
# p4: the race-day setting on the nominal car, race seeds 0-4, 3 laps (race times in s), and its cost.
P4 = {"mpc": [468.55, 468.55, 468.55, 468.65, 468.60], "pursuit": [468.55, 468.55, 468.60, 468.65, 468.60]}
P4_MAX_ERROR = {"mpc": 1.009, "pursuit": 1.660}  # m, over the five races
P4_CPU = {"mpc": 9.8, "pursuit": 3.6}  # s of computer time per race
HELD_OUT_CLEAN = {"mpc": 24, "pursuit": 13}  # of the 24 held-out cars (p3c: pursuit 11 collisions)
for steering, times in P4.items():
    print(f"{steering:8s} median {np.median(times):.2f} s, spread {max(times) - min(times):.2f} s, "
          f"max line error {P4_MAX_ERROR[steering]:.2f} m, held-out cars clean {HELD_OUT_CLEAN[steering]}/24, "
          f"{P4_CPU[steering]:.1f} CPU-s a race")
print("mpc beats pursuit:", noise_aware_better(P4["mpc"], P4["pursuit"]),
      "| pursuit beats mpc:", noise_aware_better(P4["pursuit"], P4["mpc"]))
assert not noise_aware_better(P4["mpc"], P4["pursuit"]) and not noise_aware_better(P4["pursuit"], P4["mpc"])

pursuit = race(UNIT_13, seed=34 % 5, k=34, STEERING="pursuit")
show("pursuit, car 34", pursuit)
assert not clean(pursuit) and pursuit.collisions == 1  # U13.4

fig, ax = plt.subplots(figsize=(10, 3.5))
t, error = error_trace(pursuit)
ax.plot(t, error, linewidth=0.6, label="pursuit")
for respawn in pursuit.respawn_times:
    ax.axvline(respawn, color="black", linewidth=1.0)
for limit in (-CLEAN_ERROR, CLEAN_ERROR):
    ax.axhline(limit, color="grey", linestyle="--", linewidth=1.0)
ax.set_xlabel("t (s; the solid line is the collision)")
ax.set_ylabel("error from the line (m)")
ax.set_title("pursuit at the race-day setting, held-out car 34, race seed 4")
plt.show()

On the nominal car the two steering laws are the same race to a tick or two: Unit 1's rule says neither is faster.
Unit 10's order decides: **time first, then margin**.
On time it is a tie; on margin MPC wins outright, 1.01 m against 1.66 m on the nominal car, and 24 of 24 held-out cars clean against pursuit's 13 (11 collisions, one of them car 34, above).
The setting was tuned with MPC steering, and pursuit has less margin left at the same speeds.

A race with MPC costs about 2.7 times the computer time of one with pursuit, and what the official race limits is `step()`: its 99th percentile is about 1.0 ms, inside the 5 ms budget by a factor of five.
**The submission keeps MPC.**

## 5. Qualify

`check_submission` (Unit 1) checks the file the way the official runner will load it: Python 3.8 syntax, imports, the official shape, and one race with `step()` timed.
The probe record of plan `roar-013` (p6) ran it on `unit-13.py` on its own track: **PASS**, a 464.35 s race on the check's own car, `step()` p99 0.997 ms.
The milestone runs it again; it is this unit's one timed submission check (Exercise 7's one race with the timing on is, with this file, the same race as with it off).

Its **default** track is `test-circuit`, and there the file is refused before it races (cheap: it fails as the agent is built):

In [ ]:
refused = check_submission(UNIT_13)  # the default track, test-circuit
print(refused)
assert not refused.ok and refused.step_p99_ms is None and "285" in str(refused)

The agent refuses a line solved for another track: `test-circuit` has 1,331 waypoints and needs 133 control values (one per 10 waypoints), and the file carries `monza-like`'s 285.
That refusal is a feature (Unit 9: a line on the wrong track would drive the car into a wall), and it means **a race-day file is checked on its own track**, `check_submission(UNIT_13, track="monza-like")`.
The Unit 10 lab met the same thing on CARLA with a Monza line.

**The timing budget, on purpose.**
Unit 10's agent times every `mpc()` call; a call slower than `MPC_BUDGET_MS` (5 ms) halves the horizon for the rest of the race, 40 → 20 → 10 ticks, never restored.
That protects a race on a machine you cannot check.
It also makes the race depend on the machine: Unit 12 showed four timing-on runs of one file at one seed giving four different races.
A gate can race timing off and know it is racing one exact behaviour; it cannot race every tick at which a load spike might cut the horizon in the shipped file.

So the race-day file sets **`MPC_MIN_HORIZON = MPC_HORIZON`** (40): a slow call is still **counted** in `over_budget`, but the horizon is never cut.
The race that ships is then the race the gates ran, and the timing is checked **before** the race, by `check_submission` on the race machine.
On race day you check the machine; then you **ship the behaviour you tested**.

No race is needed to see it: build two agents with `MPC_BUDGET_MS = 0.0` (every call is over budget), `unit-13.py`'s and the same file with Unit 10's `MPC_MIN_HORIZON = 10`, and call `mpc_command` three times on each:

In [ ]:
for label, minimum in (("unit-13.py, MPC_MIN_HORIZON 40", 40), ("Unit 10's policy, MPC_MIN_HORIZON 10", 10)):
    with shim_installed():
        Agent, _ = load_submission(variant(UNIT_13, MPC_BUDGET_MS=0.0, MPC_MIN_HORIZON=minimum))
    agent = Agent(track.waypoints, None)
    horizons = []
    for call in range(3):
        agent.mpc_command(0.0, 15.0)
        horizons.append(agent.horizon)
    print(f"{label}: over_budget {agent.over_budget}, the horizon after each call {horizons}")
    assert agent.over_budget == 3 and horizons == ([40, 40, 40] if minimum == 40 else [20, 10, 10])  # U13.0b

# The probe record of plan roar-013 (p9): what Unit 10's halving policy would have raced, bracketed. M2 with the
# horizon fixed at 20, and with MPC_BUDGET_MS = 0.0 (the horizon 10 from the third tick: the policy's worst case).
P9 = {"horizon 20": (5, 48, 1.38), "horizon 10 (budget 0.0)": (5, 48, 1.58)}  # (nominal clean of 5, cars 24-71 of 48, max m)
for label, (nominal, cars, error) in P9.items():
    print(f"{label:24s} nominal {nominal}/5 clean, cars 24-71 {cars}/48 clean, max line error {error:.2f} m")

With the race-day file every slow call is counted and the horizon stays at 40; with Unit 10's policy it would have been 20, then 10 (the printed lines say `unit-11`: the code is `unit-11.py`'s, unchanged).
The record also bracketed what that policy would have raced: at a fixed horizon of 20, and at 10 from the third tick, the race-day setting is clean on every nominal and validation race (106 races).
That shows each **regime** is clean, but not every **switch**: a cut can happen at any tick, and no finite set of races covers them all.
Making the policy a counter removes the question instead of sampling it.

The trade is real.
If the race machine is slower than the one `check_submission` ran on, the file no longer protects itself; that is why the check runs on the race machine, idle, before the race (the lab does).

## 6. Package and submit

Unit 12's packaging checklist, for race day:

- **one file**, `submission.py`: the official shape (`RoarCompetitionSolution`, `async def initialize` and `step`), importing only the standard library, numpy and `roar_py_interface`;
- **Python 3.8**: the official environment (design 013 §1.1); `check_submission` parses the file as Python 3.8, and the book's routed tests run it on Python 3.8 too;
- **every tuned value on its grid and written with `repr`** (Unit 12, traps 1 and 2), with a header comment saying what was changed, by which search, and why;
- **the timing as shipped** (`MPC_BUDGET_MS = 5.0`): races are compared with it off, and the race runs with it on (with `MPC_MIN_HORIZON = MPC_HORIZON` it only counts);
- **`check_submission` passes on the race track**, on the race machine, idle;
- **versions written down**: Python, numpy, racekit, and the file's SHA-256 (from the operating system's own tool, the lab says how), so you can prove later which file raced.

And **the season's rules**, which this book cannot pin for you:

- the repository you submit holds your `submission.py` and the **unmodified** `competition_runner.py` and `infrastructure.py` from the competition's fork;
- an extra dependency goes in `instruction.txt` (the series page) or `requirements.txt` (the gitbook): **the two sources disagree**, so read the current season's page, follow it, and write down what it said and when you read it (`unit-13.py` needs neither: numpy is part of the environment);
- the deadline is the season's: the current season's is 2026-12-11, and the page is where to check it.

**If the season changes something**, re-check before you trust any number in this book:

- a new `competition_runner.py`: the laps, the timeout, the collision rule and what a respawn costs (Unit 1 read the pinned one);
- a new Monza map: the waypoints, so the line (Unit 9), the sections and their grip (Unit 8) must be measured again;
- a new CARLA version or Python: the car (Unit 8 again), the Python version `check_submission` parses for, and numpy's version.

Here is the file's header, and the versions this notebook ran on:

In [ ]:
header, body = file_parts(UNIT_13)
print("".join(header))
imports = [line[:-1] for line in body if line[:7] == "import " or (line[:5] == "from " and " import " in line)]
print("imports:", imports)
assert imports == ["from __future__ import annotations", "import math", "import time",
                   "from typing import List, Optional, Sequence, Tuple", "import numpy as np", "import roar_py_interface"]
assert thirteen.MPC_BUDGET_MS == 5.0 and thirteen.STEERING == "mpc"
python = sys.version_info
print(f"Python {python[0]}.{python[1]}.{python[2]}, numpy {np.__version__}, racekit {racekit.__version__}")

**The interim CARLA check.**
Every number in this unit is racekit's.
Design 013 has the author race `agents/unit-13.py` itself on CARLA, timed, as an early warning before the book is published; it is recorded in `roar/docs/qualification.md` as **PENDING (author)** until it is run.
Your own race day on CARLA is this unit's lab.

| `monza-like`, 3 laps | `unit-07.py` (Part 1) | `unit-13.py` (race day) |
|---|---|---|
| nominal car, race seeds 0-4 | 515.10 s, 5/5 clean | **468.55 s**, 5/5 clean |
| held-out cars 24-47 (validation) | 12/24 clean | **24/24** clean |
| validation cars 48-71 | | **24/24** clean |
| test cars 72-95, raced once | | **24/24** clean |
| `check_submission` on `monza-like` | | PASS, p99 about 1.0 ms |

**What race day used, from the toolbox:**

- Unit 7's **corner cap**, to show a lower bound on grip is free on a straight;
- Units 8 and 9: the **measured grip** and the **racing line**, per track;
- Unit 12's **crash-aware search**, on the known track, with its **rounding traps** caught;
- Unit 11's **margin rule**, on every knob at its edge, checked on **perturbed cars** never tuned on, kept apart as **training, validation and test** sets;
- Unit 11's **expected race time**, to price a crash;
- Unit 10's choice of **MPC or pursuit**, time first and then margin, and its **tick budget**, turned into a counter on purpose;
- Unit 12's **packaging checklist**, and the season's rules read fresh.

Next: the **capstone**, your own competition agent, and the report that says what each stage of it bought.